In [1]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

GPU allocated: 0.0 GB


In [4]:
import shutil
import os

SRC = "/home/elicer/cineverse/gemma4-cineverse/checkpoint-3038"
BACKUP = "/home/elicer/cineverse/backup-checkpoint-3038"

if not os.path.exists(BACKUP):
    shutil.copytree(SRC, BACKUP)
    print("백업 완료:", BACKUP)
else:
    print("백업 폴더가 이미 존재합니다:", BACKUP)

백업 완료: /home/elicer/cineverse/backup-checkpoint-3038


In [5]:
import os
import shutil

CHECKPOINT_PATH = "/home/elicer/cineverse/backup-checkpoint-3038"
LORA_PATH = "/home/elicer/cineverse/gemma4-cineverse-lora-final"

if os.path.exists(LORA_PATH):
    shutil.rmtree(LORA_PATH)

os.makedirs(LORA_PATH, exist_ok=True)

files_to_copy = [
    "README.md",
    "adapter_model.safetensors",
    "adapter_config.json",
    "chat_template.jinja",
    "tokenizer_config.json",
    "tokenizer.json",
    "processor_config.json",
]

for filename in files_to_copy:
    src = os.path.join(CHECKPOINT_PATH, filename)
    dst = os.path.join(LORA_PATH, filename)

    if os.path.exists(src):
        shutil.copy2(src, dst)
        print("복사 완료:", filename)
    else:
        print("없음:", filename)

print("\n최종 LoRA 폴더:")
print(os.listdir(LORA_PATH))

복사 완료: README.md
복사 완료: adapter_model.safetensors
복사 완료: adapter_config.json
복사 완료: chat_template.jinja
복사 완료: tokenizer_config.json
복사 완료: tokenizer.json
복사 완료: processor_config.json

최종 LoRA 폴더:
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json']


In [2]:
from unsloth import FastLanguageModel

CHECKPOINT_PATH = "/home/elicer/cineverse/backup-checkpoint-3038"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CHECKPOINT_PATH,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

print("체크포인트 로드 완료")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/elicer/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.7: Fast Gemma4_Unified patching. Transformers: 5.12.1.
   \\   /|    NVIDIA A100 80GB PCIe MIG 2g.20gb. Num GPUs = 1. Max memory: 19.5 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 677/677 [00:35<00:00, 19.00it/s] 


체크포인트 로드 완료


In [3]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

print("모델 추론 모드 적용 완료")

모델 추론 모드 적용 완료


In [4]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "너는 토니 스타크다. 한 문장으로 인사해줘."
            }
        ]
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

inputs = {
    k: v.to("cuda") if hasattr(v, "to") else v
    for k, v in inputs.items()
}

outputs = model.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=False,
    use_cache=True,
)

input_length = inputs["input_ids"].shape[-1]

print(
    tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True,
    )
)

세상을 구하는 건 나에게 어울리는 일이지, 준비됐어?


In [5]:
import os
import shutil

GGUF_OUTPUT = "/home/elicer/cineverse/model/cineverse-gemma4-12b-gguf"

# 저장 폴더 준비
os.makedirs(GGUF_OUTPUT, exist_ok=True)

# 디스크 여유 확인
total, used, free = shutil.disk_usage("/home/elicer/cineverse")

print(f"전체 용량: {total / 1024**3:.1f} GB")
print(f"사용 중: {used / 1024**3:.1f} GB")
print(f"여유 공간: {free / 1024**3:.1f} GB")

전체 용량: 127.9 GB
사용 중: 59.0 GB
여유 공간: 69.0 GB


In [6]:
model.save_pretrained_gguf(
    GGUF_OUTPUT,
    tokenizer,
    quantization_method="q4_k_m",
)

print("GGUF Q4_K_M 저장 완료")
print("저장 경로:", GGUF_OUTPUT)

Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /home/elicer/cineverse/model/cineverse-gemma4-12b-gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /home/elicer/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/home/elicer/cineverse/model/cineverse-gemma4-12b-gguf`: 100%|██████████| 1/1 [02:20<00:00, 140.87s/it]


Successfully copied all 1 files from cache to `/home/elicer/cineverse/model/cineverse-gemma4-12b-gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [03:41<00:00, 221.46s/it]


Unsloth: Merge process complete. Saved to `/home/elicer/cineverse/model/cineverse-gemma4-12b-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/home/elicer/cineverse/model/cineverse-gemma4-12b-gguf_gguf/gemma-4-12b-it.BF16.gguf', '/home/elicer/cineverse/model/cineverse-gemma4-12b-gguf_gguf/gemma-4-12b-it.BF16-mmproj.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
U

In [8]:
import os

GGUF_DIR = "/home/elicer/cineverse/model/cineverse-gemma4-12b-gguf"

gguf_files = []

for root, _, files in os.walk(GGUF_DIR):
    for filename in files:
        if filename.lower().endswith(".gguf"):
            path = os.path.join(root, filename)
            size = os.path.getsize(path) / 1024**3
            gguf_files.append(path)
            print(f"{path}: {size:.2f} GB")

if not gguf_files:
    print("아직 GGUF 파일이 없습니다. 변환 셀이 완료됐는지 확인하세요.")

아직 GGUF 파일이 없습니다. 변환 셀이 완료됐는지 확인하세요.


In [9]:
ls -lh /home/elicer/cineverse/model/cineverse-gemma4-12b-gguf_gguf

total 7.2G
-rw-r--r-- 1 elicer elicer 168M Jun 16 01:42 gemma-4-12b-it.BF16-mmproj.gguf
-rw-r--r-- 1 elicer elicer 6.9G Jun 16 01:56 gemma-4-12b-it.Q4_K_M.gguf
